## kerenel to abstract data Visulize data filter for ai model.

In [4]:

# ===============================================
#  Swap Fault Sequence Preparation for LSTM (with padding for early sequences)
#  Author: Shubham
# ===============================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import joblib
# ---------- Step 1: Load CSV ----------
df = pd.read_csv("../data/swap_log_1.csv")

# ---------- Step 2: Convert hex → int ----------
def hex_to_int(x):
    try:
        if isinstance(x, str) and x.startswith("0x"):
            return int(x, 16)
        else:
            return int(x)
    except:
        return 0

df["VA"] = df["VA"].apply(hex_to_int)
df["PFN"] = df["PFN"].apply(hex_to_int)
df["mapping"] = df["mapping"].apply(hex_to_int)

# ---------- Step 3: Sort by PID and timestamp ----------
df = df.sort_values(by=["PID", "start_ns"]).reset_index(drop=True)

# ---------- Step 4: Normalize numeric features ----------
scaler = MinMaxScaler()
df[["VA", "PFN", "folio_index", "mapping", "latency_ns"]] = scaler.fit_transform(
    df[["VA", "PFN", "folio_index", "mapping", "latency_ns"]]
)

# ---------- Step 5: Create fixed-length sequences (with padding for shorter sequences) ----------
SEQ_LEN = 10  # number of past events used for prediction
X, y = [], []

for pid in df["PID"].unique():
    pid_data = df[df["PID"] == pid][["VA", "PFN", "folio_index", "mapping", "latency_ns"]].values
    n = len(pid_data)

    # Iterate over each entry as target
    for i in range(n):
        start_idx = max(0, i - SEQ_LEN)
        seq = pid_data[start_idx:i]

        # pad if sequence smaller than SEQ_LEN
        if len(seq) < SEQ_LEN:
            pad = np.zeros((SEQ_LEN - len(seq), pid_data.shape[1]))
            seq = np.vstack((pad, seq))

        X.append(seq)
        y.append(pid_data[i])  # current entry is target

X = np.array(X)
y = np.array(y)

print("✅ Data prepared for LSTM training (with padding for early entries)")
print(f"Input shape (X): {X.shape}")
print(f"Target shape (y): {y.shape}")

# ---------- Step 6: Save preprocessed data ----------
np.save("X_swap_seq.npy", X)
np.save("y_swap_seq.npy", y)

print("Saved: X_swap_seq.npy & y_swap_seq.npy")

# ---------- Step 7 (optional): Show one example ----------
print("\nExample sequence (first sample):")
print(X[5])
print("\nNext target (y[0]):")
print(y[5])

joblib.dump(scaler, 'minmax_scaler.pkl')

✅ Data prepared for LSTM training (with padding for early entries)
Input shape (X): (1390, 10, 5)
Target shape (y): (1390, 5)
Saved: X_swap_seq.npy & y_swap_seq.npy

Example sequence (first sample):
[[0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.67294044 0.50737602 0.         0.         0.00466323]
 [0.67294044 0.06240519 0.00288184 0.         0.00472239]
 [0.67294044 0.28472879 0.00576369 0.         0.00473912]
 [0.67294044 0.16813042 0.00864553 0.         0.00475132]
 [0.67294044 0.06242181 0.01152738 0.         0.00476376]]

Next target (y[0]):
[0.67294044 0.67916459 0.01440922 0.         0.00477686]


['minmax_scaler.pkl']

In [6]:

# ===============================================
#  Interactive Swap Fault Trend Visualization
#  Author: Shubham
# ===============================================

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# ---------- Step 1: Load Data ----------
df = pd.read_csv("swap_log.csv")
print("Data loaded:", df.shape)
df['latency_sec'] = round(df['latency_ns']/1000000000,3)
df['start_sec'] = round(df['start_ns']/1000000000,3)
# ---------- Step 2: Convert hex → int ----------
def hex_to_int(x):
    try:
        if isinstance(x, str) and x.startswith("0x"):
            return int(x, 16)
        else:
            return int(x)
    except:
        return np.nan

df["VA"] = df["VA"].apply(hex_to_int)
df["PFN"] = df["PFN"].apply(hex_to_int)
df["mapping"] = df["mapping"].apply(hex_to_int)

df = df.sort_values(by=["PID", "start_sec"]).reset_index(drop=True)

# ---------- Step 3: Summary ----------
print("\nProcesses found:", df["PID"].nunique())
print("Top 5 processes by entry count:\n", df["PID"].value_counts().head())

# ---------- Step 4: Latency Distribution ----------
fig1 = px.histogram(df, x="latency_sec", nbins=60, title="Latency Distribution (sec)",
                    marginal="box", template="plotly_dark", color_discrete_sequence=["#1f77b4"])
fig1.update_layout(xaxis_title="Latency (sec)", yaxis_title="Frequency")
fig1.show()

# ---------- Step 5: Latency Over Time (Filterable by PID) ----------
fig2 = px.scatter(df, x="start_sec", y="latency_sec", color=df["PID"].astype(str),
                  hover_data=["COMM", "VA", "PFN", "folio_index"],
                  title="Latency Over Time (Group by PID)",
                  template="plotly_dark", opacity=0.8)
fig2.update_layout(xaxis_title="start_sec (time)", yaxis_title="latency_sec (sec)")
fig2.show()

# ---------- Step 6: PFN vs Latency ----------
fig3 = px.scatter(df, x="PFN", y="latency_sec",
                  size=np.sqrt(df["latency_sec"])*10,  # make high-latency points larger
                  color=df["COMM"],
                  hover_data=["PID", "VA", "folio_index"],
                  title="PFN vs Latency (Seconds, scaled by process)",
                  template="plotly_dark")

fig3.update_layout(xaxis_title="PFN", yaxis_title="Latency (sec)")
fig3.show()

# ---------- Step 7: Folio Index vs Latency ----------
fig4 = px.box(df, x="COMM", y="latency_sec",
              color="COMM", title="Latency per Process (Box Plot)",
              template="plotly_dark")
fig4.update_layout(xaxis_title="Process Name", yaxis_title="Latency (sec)")
fig4.show()

# ---------- Step 8: Correlation Heatmap ----------
import plotly.figure_factory as ff

numeric_cols = ["VA", "PFN", "folio_index", "mapping", "latency_sec"]
corr = df[numeric_cols].corr()

fig5 = ff.create_annotated_heatmap(
    z=corr.values,
    x=list(corr.columns),
    y=list(corr.index),
    annotation_text=corr.round(2).values,
    colorscale="RdBu",
    showscale=True
)
fig5.update_layout(title="Feature Correlation Heatmap", template="plotly_dark")
fig5.show()



Data loaded: (1263, 8)

Processes found: 14
Top 5 processes by entry count:
 PID
15494    200
15561    200
16097    200
16150    200
16988    200
Name: count, dtype: int64
